In [ ]:
# Install Java and Spark in Colab
!apt-get update
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q http://www.apache.org/dist/spark/spark-3.4.0/spark-3.4.0-bin-hadoop3.tgz
!tar xf spark-3.4.0-bin-hadoop3.tgz
!pip install -q findspark
!pip install pyspark
!wget -q https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/2-adverse-events.csv
!ls -lh 2-adverse-events.csv

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
tar: spark-3.4.0-bin-hadoop3.tgz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now
-rw-r--r-- 1 root root 588K Feb 23 21:27 2-adverse-events.csv


In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, Row
import collections

In [ ]:
# Initialize a SparkSession
session = SparkSession.builder.appName("DataFrameOperations").getOrCreate()

def normalize(name):
  name = name.replace("_", " ") # replace underscores with spaces
  name = name.strip().upper() # remove spaces and make it to uppercase
  name = " ".join(name.split()) # collapse multiple spaces
  return name

def parseLine(filterLine):
  # Split the line by commas
  fields = filterLine.split(',')
  # Extract the adverse_event and arm_affected
  Adverse-Event == normalize(fields[1])
  # Return a tuple (adverse_event, arm_affected)
  return (Adverse-Event)

# Read the CSV file into a DataFrame using SparkSession.read.csv
# Assuming the CSV has a header and we want Spark to infer the schema
inputDF = session.read.csv("2-adverse-events.csv", header=True, inferSchema=True).cache()

# Create a temporary view from the DataFrame
inputDF.createOrReplaceTempView("AdverseEvents")

# Use SQL query to create specific view
session.sql("SELECT * FROM AdverseEvents").Ashow()

+-----------+--------------------+------------+-----------+--------------------+-----------+--------------------+--------------------+--------------------+--------------+----+--------------------+----------+----------------+--------------------+
|allergic_ID|       Adverse-Event|Arm-affected|Arm-at-risk|            Category|   Trial-Id|                 URL|           Condition|        Intervention| Serious-Other|Arm.|           Arm_Title|classes_ID|            Drug|               Class|
+-----------+--------------------+------------+-----------+--------------------+-----------+--------------------+--------------------+--------------------+--------------+----+--------------------+----------+----------------+--------------------+
|       1107|Drug_hypersensiti...|           2|        328|Immune_system_dis...|NCT00001213|https://clinicalt...|          Cystinosis|          Cysteamine|  other_events|arm0|Cysteamine-Topica...|      NULL|            NULL|                NULL|
|       2291|   

In [ ]:
# Create temp table view that contains "doxorubichin" in "drug" column
session.sql("SELECT * FROM AdverseEvents WHERE Drug = 'doxorubicin'").show()

+-----------+--------------------+------------+-----------+--------------------+-----------+--------------------+--------------------+--------------------+--------------+----+--------------------+----------+-----------+--------------------+
|allergic_ID|       Adverse-Event|Arm-affected|Arm-at-risk|            Category|   Trial-Id|                 URL|           Condition|        Intervention| Serious-Other|Arm.|           Arm_Title|classes_ID|       Drug|               Class|
+-----------+--------------------+------------+-----------+--------------------+-----------+--------------------+--------------------+--------------------+--------------+----+--------------------+----------+-----------+--------------------+
|       2293|         Anaphylaxis|           1|       1748|Immune_system_dis...|NCT00003782|https://clinicalt...|       Breast_Cancer|cyclophosphamide|...|serious_events|arm0|Doxorubicin-+-Cyc...|        38|doxorubicin|anthracycline-ant...|
|       2292|         Anaphylaxis|  

In [ ]:
# total affected patients with the "Hypersensitivity" event
session.sql("SELECT Count(*) FROM AdverseEvents WHERE `Adverse-Event`= 'Hypersensitivity'").show()

+--------+
|count(1)|
+--------+
|     755|
+--------+



In [ ]:
# how many unique trials?
session.sql("SELECT Count(DISTINCT `Trial-Id`) FROM AdverseEvents").show()

+------------------------+
|count(DISTINCT Trial-Id)|
+------------------------+
|                     794|
+------------------------+



In [ ]:
# for each unique trial, how many unique arms?
session.sql("SELECT `Trial-Id`, COUNT(DISTINCT `Arm.`) AS UniqueArms FROM AdverseEvents GROUP BY `Trial-Id` ORDER BY `Trial-Id`").show()

+-----------+----------+
|   Trial-Id|UniqueArms|
+-----------+----------+
|NCT00001213|         1|
|NCT00003298|         1|
|NCT00003389|         1|
|NCT00003782|         2|
|NCT00003869|         1|
|NCT00003896|         1|
|NCT00004859|         2|
|NCT00004888|         2|
|NCT00006011|         2|
|NCT00006721|         2|
|NCT00008138|         1|
|NCT00010257|         1|
|NCT00022659|         1|
|NCT00025155|         1|
|NCT00025233|         1|
|NCT00027027|         1|
|NCT00036270|         2|
|NCT00038103|         2|
|NCT00038467|         2|
|NCT00039130|         1|
+-----------+----------+
only showing top 20 rows


In [ ]:
session.stop()